# Proceso para generar la capa Silver

La capa Silver en la arquitectura medallion es responsable de limpiar y validar los datos provenientes de la capa Bronze (datos crudos). El proceso típico incluye:

1. **Ingesta desde Bronze:** Se leen los datos sin procesar almacenados en la capa Bronze.
2. **Limpieza de datos:** Se eliminan o corrigen registros nulos, duplicados o inválidos.
3. **Validación:** Se aplican reglas de negocio para asegurar la calidad y consistencia de los datos.
4. **Transformaciones:** Se pueden realizar uniones, enriquecimientos y cambios de formato según sea necesario.
5. **Almacenamiento:** Los datos refinados se guardan en tablas de la capa Silver, listos para análisis más avanzados o para alimentar la capa Gold.

Esta capa sirve como base confiable para análisis y modelado de datos.

## 1. Configuración e Importación de Librerías

In [0]:
# Celda 1: Recepción de Parámetros
schema_name = dbutils.widgets.text("schema_name", "")
schema_name = dbutils.widgets.get("schema_name")

bronze_table = dbutils.widgets.text("bronze_table", "")
bronze_table = dbutils.widgets.get("bronze_table")

silver_table = dbutils.widgets.text("silver_table", "")
silver_table = dbutils.widgets.get("silver_table")


In [0]:
# Librerias de manipulación de datos
from pyspark.sql import functions as F
from pyspark.errors import AnalysisException

In [0]:

# generar nombre completo de la tabla
def qname(table):
    return f"{schema_name}.{table}"

BRONZE_FULL = qname(bronze_table)
SILVER_FULL = qname(silver_table)
print("Tabla Bronze:", BRONZE_FULL)
print("Tabla Silver:", SILVER_FULL)

## 2. Diccionario de Columnas del Dataset

### LPN CASE HEADER (CH) - 9 columnas

| Columna | Descripción |
|---------|-------------|
| `CASE_NBR` | CH Nro Caja |
| `DC_ORD_NBR` | CH Nro Centro Distribucion |
| `CONS_PRTY_DATE` | CH Fecha Prioridad de Consumo |
| `PROC_IMMD_NEEDS` | CH Process in Immediate Needs |
| `VOL` | CH Volumen |
| `EST_WT` | CH Peso Estimado |
| `CREATE_DATE_TIME` | CH Fecha Creacion |
| `RCVD_DATE` | CH Fecha Recepcion |
| `SNGL_SKU_CASE` | CH Solo un SKU |

### LPN CASE DETAIL (CD) - 2 columnas

| Columna | Descripción |
|---------|-------------|
| `ORIG_QTY` | CD Cant Original |
| `STD_PACK_QTY` | CD Cant Std Empaque |

### ITEM MASTER (IM) - 9 columnas

| Columna | Descripción |
|---------|-------------|
| `PROD_TYPE` | IM Material Handv |
| `UNIT_PRICE` | IM Unit Price |
| `STD_CASE_QTY` | IM Std Case Qty |
| `STORE_DEPT` | IM Division PMM |
| `MERCH_TYPE` | IM Dpto PMM |
| `SPL_INSTR_CODE_3` | IM Special Inst Code 3 |
| `ECCN_NBR` | IM Codigo Temporada |
| `ORGN_CERT_CODE` | IM Origination Certification Code |
| `MV_SIZE_UOM` | IM Area PMM |

### LOCATION HEADER (LH)

| Columna | Descripción |
|---------|-------------|
| `ZONE` | LH Zona |
| `SKU_DEDCTN_TYPE` | LH Tipo de Dedicacion |
| `SLOT_UNUSABLE` | LH Slot Info Unusable Locn |

---
**Total: 23 columnas documentadas**

## 2. Cargar el conjunto de `datos`

In [0]:

# Leer datos de la tabla Bronze
lpn_bronze = spark.table(BRONZE_FULL)
display(lpn_bronze.limit(10))
# Dimensiones del dataset
print(f"Filas: {lpn_bronze.count()}, Columnas: {len(lpn_bronze.columns)}")

In [0]:
lpn_bronze.printSchema()

## 3. Correccion del tipo de datos

In [0]:
# correcion de tipo de datos
cols_to_cast = [
    "DC_ORD_NBR", "STORE_DEPT", "MERCH_TYPE", "ECCN_NBR", "MV_SIZE_UOM"
]

for c in cols_to_cast:
    lpn_bronze = lpn_bronze.withColumn(c, F.col(c).cast("string")) 

In [0]:
lpn_bronze.printSchema()

## 4. Eliminacion de duplicados

In [0]:
# eliminacion de duplicados que representen menos del 1% del dataset sino dejarlo intacto
total = lpn_bronze.count()
dupes = lpn_bronze.count() - lpn_bronze.dropDuplicates().count()
porcentaje = dupes / total

if porcentaje < 0.01:
    lpn_bronze = lpn_bronze.dropDuplicates()
    print(f"Duplicados eliminados. {dupes} de {total}")
else:
    print("Duplicados conservados (más del 1%).")
print(f"Total: {total}, Duplicados: {dupes}, Porcentaje: {porcentaje}") 
print(f"Filas: {lpn_bronze.count()}, Columnas: {len(lpn_bronze.columns)}")

## 5. Eliminacion de variables
Se eliminaran Variables de tipo identificadoras, variables categóricas con una sola categoría dominante y la fecha de creacion.

In [0]:
# Detectar columnas categóricas (tipo string)
dtypes = dict(lpn_bronze.dtypes)
cat_cols = [c for c, t in dtypes.items() if t == "string"]

# Calcular cardinalidad y porcentaje dominante para cada columna
stats = []
total = lpn_bronze.count()

for col in cat_cols:
    freq_df = (
        lpn_bronze.groupBy(col)
        .count()
        .orderBy(F.desc("count"))
    )
    cardinalidad = freq_df.count()
    row = freq_df.first()
    if row:
        clase_dominante = row[col]
        porcentaje = row["count"] / total * 100
    else:
        clase_dominante = None
        porcentaje = None
    stats.append((col, cardinalidad, clase_dominante, porcentaje))

# Crear DataFrame de resultados
result_df = spark.createDataFrame(stats, ["columna", "cardinalidad", "clase_dominante", "porcentaje_dominante"])
display(result_df.orderBy(F.desc("porcentaje_dominante")))

In [0]:
cols_to_drop = ["CASE_NBR", "SNGL_SKU_CASE", "SKU_DEDCTN_TYPE", "SLOT_UNUSABLE", "CREATE_DATE_TIME"]
lpn_bronze = lpn_bronze.drop(*cols_to_drop)

In [0]:
# validar borrado exitoso
assert lpn_bronze.columns == [c for c in lpn_bronze.columns if c not in cols_to_drop], "Columnas no eliminadas correctamente"

## 6. Tratamiento de valores nulos

In [0]:
# buscar valores nulos solo mostrar columnas nulas
df_nulls = lpn_bronze.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in lpn_bronze.columns])
df_nulls.show(vertical=True)

In [0]:


# Contar valores nulos por fila
null_count_col = sum([F.when(F.col(c).isNull(), 1).otherwise(0) for c in lpn_bronze.columns])

# Filtrar filas con 10 o menos columnas nulas
lpn_bronze = lpn_bronze.withColumn("null_count", null_count_col) \
             .filter(F.col("null_count") <= 10) \
             .drop("null_count")

# Mostrar cuántas filas se eliminaron
print(f"Filas originales: {lpn_bronze.count()}")
print(f"Filas después de filtrar: {lpn_bronze.count()}")


In [0]:
fill_dict = {"DC_ORD_NBR": "unknown", "CONS_PRTY_DATE": "1900-01-01", "RCVD_DATE": "1900-01-01"}
lpn_bronze = lpn_bronze.fillna(fill_dict)
# Validar que se hayan rellenado los valores nulos
for c in fill_dict.keys():
    assert lpn_bronze.filter(F.col(c).isNull()).count() == 0, f"No se han rellenado los valores nulos en la columna {c}"


## 7. Normalización de strings

In [0]:
# Normalización de strings trim, lower a todas las columnas categoricas
dtypes = dict(lpn_bronze.dtypes)
cat_cols = [c for c, t in dtypes.items() if t == "string"]

for c in cat_cols:
    lpn_bronze = lpn_bronze.withColumn(c, F.lower(F.trim(F.col(c))))
# Eliminar espacios en blanco de las columnas categoricas
for c in cat_cols:
    lpn_bronze = lpn_bronze.withColumn(c, F.regexp_replace(F.col(c), ' ', '_'))
# Eliminar caracteres especiales de las columnas categoricas
for c in cat_cols:
    lpn_bronze = lpn_bronze.withColumn(c, F.regexp_replace(F.col(c), '[^a-zA-Z0-9_]', ''))


## 8. Extracción de Features Temporales

In [0]:

# Definir las columnas de fecha
date_cols = ["CONS_PRTY_DATE", "RCVD_DATE"]

# Crear nuevas columnas para cada componente de la fecha
for colname in date_cols:
    lpn_bronze = lpn_bronze.withColumn(f"{colname}_year", F.year(F.col(colname))) \
        .withColumn(f"{colname}_month", F.month(F.col(colname))) \
        .withColumn(f"{colname}_day", F.dayofmonth(F.col(colname))) \
        .withColumn(f"{colname}_day_of_week", F.dayofweek(F.col(colname)))
    
    # Solo extraer hora para RCVD_DATE, no para CONS_PRTY_DATE
    if colname == "RCVD_DATE":
        lpn_bronze = lpn_bronze.withColumn(f"{colname}_hour", F.hour(F.col(colname)))

## 9. Agrupacion de categorias con baja representacion

In [0]:
low_freq_cols = ["MERCH_TYPE", "MV_SIZE_UOM"]
total = lpn_bronze.count()
for c in low_freq_cols:
    freq = lpn_bronze.groupBy(c).count().withColumn('pct', (F.col('count') / total) * 100)
    low = freq.filter(F.col("pct") < F.lit(0.4))\
              .select(F.col(c).alias("key"))
    lpn_bronze = (lpn_bronze.join(low, lpn_bronze[c]==low["key"], "left")
            .withColumn(c, F.when(F.col("key").isNotNull(), F.lit("otros")).otherwise(F.col(c)))
            .drop("key"))

## 10. Guardar el conjunto de `datos` en capa silver

In [0]:
# Escribimos el DataFrame en formato Delta y lo registramos como tabla
lpn_bronze.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(SILVER_FULL)

In [0]:
# Verifica conteos y muestra esquema
try:
    df_check = spark.table(SILVER_FULL)
    print("Filas Silver:", df_check.count())
    df_check.printSchema()
    display(df_check.limit(10))
except AnalysisException as e:
    print("Silver aún no materializado (si estás en streaming, espera al menos 1 microbatch).", str(e))